# Example 1.2 Diabetes Regression

**Objective:** Complete regression example with MLflow logging.

This example is independent and can run directly.

In [ ]:


import os
import sys
import json
import time
import math
import pickle
import tempfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mlflow
import mlflow.sklearn
import mlflow.pyfunc
from mlflow.tracking import MlflowClient
from mlflow.models.signature import infer_signature

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.datasets import (
    load_iris, load_diabetes, load_digits, load_linnerud,
    load_wine, load_breast_cancer, make_classification, make_regression
)
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge, ElasticNet
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.multioutput import MultiOutputRegressor

warnings.filterwarnings("ignore")


def find_project_root(start_path=None):
    """Find the mlflow_for_ml_dev folder from wherever Jupyter is started."""
    start = Path(start_path or Path.cwd()).resolve()
    for parent in [start] + list(start.parents):
        if parent.name == "mlflow_for_ml_dev" and (parent / "notebooks").exists():
            return parent
        if (parent / "datasets").exists() and (parent / "mlruns").exists() and (parent / "notebooks").exists():
            return parent
    # Fallback: if user started Jupyter inside notebooks subfolder
    for parent in [start] + list(start.parents):
        if parent.name == "notebooks":
            return parent.parent
    return start

PROJECT_ROOT = find_project_root()
DATASETS_DIR = PROJECT_ROOT / "datasets"
MLRUNS_DIR = PROJECT_ROOT / "mlruns"
MODELS_DIR = PROJECT_ROOT / "models"
ARTIFACTS_DIR = PROJECT_ROOT / "notebooks" / "artifacts_generated"

for folder in [DATASETS_DIR, MLRUNS_DIR, MODELS_DIR, ARTIFACTS_DIR, MLRUNS_DIR / ".trash"]:
    folder.mkdir(parents=True, exist_ok=True)

# IMPORTANT: Every notebook logs to the same mlruns folder inside this project.
mlflow.set_tracking_uri(MLRUNS_DIR.as_uri())
client = MlflowClient()

print("Project root      :", PROJECT_ROOT)
print("Datasets folder   :", DATASETS_DIR)
print("MLflow runs folder:", MLRUNS_DIR)
print("Models folder     :", MODELS_DIR)
print("Tracking URI      :", mlflow.get_tracking_uri())


## 1. Setup

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score

X, y = load_diabetes(as_frame=True, return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)
model = Ridge(alpha=1.0).fit(X_train, y_train)
preds = model.predict(X_test)
rmse = mean_squared_error(y_test, preds) ** 0.5
r2 = r2_score(y_test, preds)

mlflow.set_experiment("independent_example_diabetes")
with mlflow.start_run(run_name="diabetes_ridge") as run:
    mlflow.log_param("alpha", 1.0)
    mlflow.log_metrics({"rmse": rmse, "r2": r2})
    mlflow.sklearn.log_model(model, "model", input_example=X_test.head(3))
    print("RMSE:", rmse, "R2:", r2, "Run:", run.info.run_id)